In [1]:
# ==============================================================================
# PROJETO: Otimização de Consultas e Joins em PySpark
# ==============================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast

spark = SparkSession.builder \
    .appName("OtimizacaoJoinsPySpark") \
    .getOrCreate()

# ------------------------------------------------------------------------------
# Etapa 1: Leia o arquivo 'videos-preparados.snappy.parquet' no dataframe 'df_video'
# ------------------------------------------------------------------------------
df_video = spark.read.parquet("videos-preparados.snappy.parquet")

# ------------------------------------------------------------------------------
# Etapa 2: Leia o arquivo 'videos-comments-tratados.snappy.parquet' no dataframe 'df_comments'
# (Nota: verificando o nome real do arquivo na pasta trata_dev: videos-comments-tratados.snappy.parquet)
# ------------------------------------------------------------------------------
import os
caminho_comments = "videos-comments-tratados.snappy.parquet" if os.path.exists("videos-comments-tratados.snappy.parquet") else "video-comments-tratados.snappy.parquet"
df_comments = spark.read.parquet(caminho_comments)

print("--- Schemas Carregados ---")
df_video.printSchema()
df_comments.printSchema()

c:\Users\Paulo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


--- Schemas Carregados ---
root
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Keyword Index: double (nullable = true)
 |-- Features PCA: vector (nullable = true)
 |-- Features Normal: vector (nullable = true)
 |-- Features: vector (nullable = true)

root
 |-- Video ID: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)
 |-- Year: string (nullable = true)
 |-- Comment: string (nullable = true

In [2]:
# ------------------------------------------------------------------------------
# Etapa 3: Crie tabelas temporárias para ambos os dataframes
# ------------------------------------------------------------------------------
df_video.createOrReplaceTempView("tb_video")
df_comments.createOrReplaceTempView("tb_comments")

In [3]:
# ------------------------------------------------------------------------------
# Etapa 4: Faça um join das tabelas criadas anteriormente utilizando o spark.sql
# no dataframe 'join_video_comments'
# ------------------------------------------------------------------------------
# Evitamos ambiguidade de colunas repetidas selecionando explícita ou distintamente
join_video_comments = spark.sql("""
    SELECT 
        v.*,
        c.Comment,
        c.`Likes Comment`,
        c.Sentiment
    FROM tb_video v
    INNER JOIN tb_comments c
        ON v.`Video ID` = c.`Video ID`
""")

print(f"Total de registros no join padrão: {join_video_comments.count()}")
join_video_comments.show(5)

Total de registros no join padrão: 18409
+--------------------+-----------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+--------------------+-------------+---------+
|               Title|   Video ID|Published At|Keyword|Likes|Comments| Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|             Comment|Likes Comment|Sentiment|
+--------------------+-----------+------------+-------+-----+--------+------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+--------------------+-------------+---------+
|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|    8|         23.0|[0.5461641657286636]|[2.07229197864298...|[3407.0,135612.0,...|Let's not forget ...|           95|        1|
|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech| 3407|     6

In [4]:
# ------------------------------------------------------------------------------
# Etapa 5: Faça as mesmas etapas anteriores (1, 2, 3, 4) utilizando repartition e coalesce
# ------------------------------------------------------------------------------
# 1 e 2. Leitura com redistribuição de partições (repartition aumenta/equilibra paralelismo antes do join)
df_video_rep = spark.read.parquet("videos-preparados.snappy.parquet").repartition(4, "Video ID")
df_comments_rep = spark.read.parquet(caminho_comments).repartition(4, "Video ID")

# 3. Criação das views temporárias particionadas
df_video_rep.createOrReplaceTempView("tb_video_rep")
df_comments_rep.createOrReplaceTempView("tb_comments_rep")

# 4. Join via Spark SQL seguido de coalesce para reduzir o número de partições de saída sem shuffle extra
join_video_comments_rep = spark.sql("""
    SELECT 
        v.*,
        c.Comment,
        c.`Likes Comment`,
        c.Sentiment
    FROM tb_video_rep v
    INNER JOIN tb_comments_rep c
        ON v.`Video ID` = c.`Video ID`
""").coalesce(1)

print(f"Número de partições após coalesce: {join_video_comments_rep.rdd.getNumPartitions()}")
print(f"Total de registros (repartition + coalesce): {join_video_comments_rep.count()}")

Número de partições após coalesce: 1
Total de registros (repartition + coalesce): 18409


In [5]:
# ------------------------------------------------------------------------------
# Etapa 6: Utilize o explain para entender melhor as duas formas de realizar as etapas
# e refaça novamente as etapas anteriores (1, 2, 3, 4), utilizando tudo que você já aprendeu
# para realizar o join e filter apenas com os dados necessários.
# ------------------------------------------------------------------------------

# 1. Inspeção dos planos de execução (Physical Plan) das abordagens anteriores:
# O explain() mostra como o Catalyst Optimizer do Spark executa o BroadcastHashJoin ou SortMergeJoin
# e onde ocorrem trocas de dados na rede (Exchange / Shuffle).
print("=== EXPLAIN: JOIN PADRÃO ===")
join_video_comments.explain(mode="formatted")

print("=== EXPLAIN: JOIN COM REPARTITION E COALESCE ===")
join_video_comments_rep.explain(mode="formatted")

# 2. Refazendo as etapas (1, 2, 3, 4) aplicando as melhores práticas de otimização:
# - Column Pruning (seleção antecipada apenas das colunas necessárias antes do join, reduzindo I/O e memória)
# - Predicate Pushdown (filtragem antecipada de registros nulos e válidos antes de cruzar as tabelas)
# - Broadcast Hash Join (como df_video tem ~1.869 linhas e é bem menor que df_comments, enviamos
#   df_video inteiro para a memória dos executores via BROADCAST, eliminando o Shuffle pesado na rede)

# Leitura filtrada e projetada (apenas colunas úteis e sem chaves nulas)
df_video_opt = spark.read.parquet("videos-preparados.snappy.parquet") \
    .select("Video ID", "Title", "Published At", "Keyword", "Likes", "Comments", "Views", "Year", "Month") \
    .filter(col("Video ID").isNotNull())

df_comments_opt = spark.read.parquet(caminho_comments) \
    .select("Video ID", "Comment", "Likes Comment", "Sentiment") \
    .filter(col("Video ID").isNotNull() & col("Comment").isNotNull())

# Criação das views temporárias otimizadas
df_video_opt.createOrReplaceTempView("tb_video_opt")
df_comments_opt.createOrReplaceTempView("tb_comments_opt")

# Execução do Join otimizado via Spark SQL usando a dica /*+ BROADCAST(v) */ e coalesce(1) para saída compacta
join_otimizado = spark.sql("""
    SELECT /*+ BROADCAST(v) */
        v.`Video ID`,
        v.Title,
        v.`Published At`,
        v.Keyword,
        v.Likes,
        v.Comments,
        v.Views,
        v.Year,
        v.Month,
        c.Comment,
        c.`Likes Comment`,
        c.Sentiment
    FROM tb_video_opt v
    INNER JOIN tb_comments_opt c
        ON v.`Video ID` = c.`Video ID`
""").coalesce(1)

print("=== EXPLAIN: JOIN OTIMIZADO (PRUNING + PUSHDOWN + BROADCAST) ===")
join_otimizado.explain(mode="formatted")
join_otimizado.show(5)

=== EXPLAIN: JOIN PADRÃO ===
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin Inner BuildLeft (6)
      :- BroadcastExchange (3)
      :  +- Filter (2)
      :     +- Scan parquet  (1)
      +- Filter (5)
         +- Scan parquet  (4)


(1) Scan parquet 
Output [14]: [Title#0, Video ID#1, Published At#2, Keyword#3, Likes#4, Comments#5, Views#6, Interaction#7, Year#8, Month#9, Keyword Index#10, Features PCA#11, Features Normal#12, Features#13]
Batched: true
Location: InMemoryFileIndex [file:/d:/Redes Sociais/Portifólio/trata_dev/videos-preparados.snappy.parquet]
PushedFilters: [IsNotNull(`Video ID`)]
ReadSchema: struct<Title:string,Video ID:string,Published At:date,Keyword:string,Likes:int,Comments:int,Views:int,Interaction:int,Year:int,Month:int,Keyword Index:double,Features PCA:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,Features Normal:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>,Features:struct<type:t

In [6]:
# ------------------------------------------------------------------------------
# Etapa 7: Salve o seu join otimizado como 'join-videos-comments-otimizado' no formato parquet
# ------------------------------------------------------------------------------
# Comentário das ações realizadas na gravação:
# 1. Tentamos primeiramente a gravação nativa distribuída do PySpark com mode('overwrite'),
#    que salva o DataFrame otimizado no diretório/arquivo 'join-videos-comments-otimizado' em Parquet (compressão Snappy padrão).
# 2. Caso o ambiente Windows local não possua o binário winutils.exe (HADOOP_HOME) configurado,
#    o bloco except aciona automaticamente o fallback seguro via Pandas/PyArrow para gerar o Parquet
#    sem travar a execução do notebook.

try:
    join_otimizado.write.mode("overwrite").parquet("join-videos-comments-otimizado")
    print("Arquivo 'join-videos-comments-otimizado' salvo com sucesso via Spark!")
except Exception as e:
    print("Aviso: HADOOP_HOME ausente no Windows para gravação nativa Spark. Salvando via PyArrow/Pandas...")
    join_otimizado.toPandas().to_parquet("join-videos-comments-otimizado.snappy.parquet", index=False)
    print("Arquivo 'join-videos-comments-otimizado.snappy.parquet' salvo com sucesso!")

Aviso: HADOOP_HOME ausente no Windows para gravação nativa Spark. Salvando via PyArrow/Pandas...


c:\Users\Paulo\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Arquivo 'join-videos-comments-otimizado.snappy.parquet' salvo com sucesso!
